In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
from dash import dash_table as dt
import plotly.express as px
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD
from python_module_one import AnimalShelter  

# Database
username = "aacuser"
password = "userpassword835"
shelter = AnimalShelter(username, password)

# Retreive Database 
data = shelter.read({'name': ''})

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({'name': ''})).drop('_id', axis = 1)
df = df.sort_values(by=['rec_num'])
rescue_types = ['Water Rescue', 'Mountain or Wilderness Rescue', 'Disaster or Individual Tracking', 'Reset']


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Graz.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Title of class, image, and name
    html.Center(children=[
        html.B(html.H1('Dashboard')),
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '300px'}),
        # Unique identifier
        html.Center(html.H4('')),
    ]),
    html.Hr(),
    
    # Each filter
    dcc.RadioItems(
        id='rescue-type-filter',
        options=[
            {'label': 'Reset', 'value': 'Reset'},
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'}
        ],
        value='Reset',
        labelStyle={'display': 'inline-block'}
    ),
    html.Hr(),
    dt.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),

        # Features for the interactive data table to make it user-friendly for the client
        editable=False,
        filter_action="native",
        sort_action="native",
        column_selectable=False,
        row_selectable=False,
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_size=12,
        page_current=0,
    ),
    html.Br(),
    html.Hr(),
    
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################

# Filter based on rescue type
@app.callback(Output('datatable-id','data'),
              [Input('rescue-type-filter', 'value')])
def dashboard(filter_type):
    
    dff = df.copy()

    if filter_type == 'Reset':
        return dff.to_dict('records')
    elif filter_type == 'Water Rescue':
        breeds = ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']
        dff = dff[dff['breed'].isin(breeds) & 
                  (dff['sex_upon_outcome'] == 'Intact Female') & 
                  (dff['age_upon_outcome_in_weeks'] >= 26) & 
                  (dff['age_upon_outcome_in_weeks'] <= 156)]
    elif filter_type == 'Mountain or Wilderness':
        breeds = ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']
        dff = dff[dff['breed'].isin(breeds) & 
                  (dff['sex_upon_outcome'] == 'Intact Male') & 
                  (dff['age_upon_outcome_in_weeks'] >= 26) & 
                  (dff['age_upon_outcome_in_weeks'] <= 156)]
    elif filter_type == 'Disaster or Individual Tracking':
        breeds = ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']
        dff = dff[dff['breed'].isin(breeds) & 
                  (dff['sex_upon_outcome'] == 'Intact Male') & 
                  (dff['age_upon_outcome_in_weeks'] >= 20) & 
                  (dff['age_upon_outcome_in_weeks'] <= 300)]

    return dff.to_dict('records')

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Pie Chart for


# Map callback that displays markers for the locations of animals from them viewport data
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_viewport_data")])
def update_map(viewData):
    dff = pd.DataFrame.from_dict(viewData)
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id")] +
            # Marker with tool tip and popup
            [dl.Marker(position=[row['location_lat'],row['location_long']], children=[
                dl.Tooltip(row['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(row['name'])
                ])
            ]) for index, row in dff.iterrows()]
        )
    ]

# Pie Chart that shows animal breed
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_viewport_data")])
def graph(viewData):
    dff = pd.DataFrame.from_dict(viewData)
    df_grouped = dff.groupby(['breed']).size().reset_index(name='Amount')
    df_grouped = df_grouped.rename(columns = {'1': 'Amount'})
    
    fig = px.pie(df_grouped, values='Amount', names='breed', title="Animal Breed")
    return [
        dcc.Graph(figure=fig)
    ]

app.run_server(debug=True)